# Grounding DINO

Change the recorded prompt or threshold. Which objects disappear first?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/IDEA-Research/grounding-dino-tiny

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os, random, requests, tarfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    SamModel, SamProcessor,
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
GDINO_ID = "IDEA-Research/grounding-dino-tiny"
print(f"📦 Grounding DINO: {GDINO_ID}")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(device)
gdino_model.eval()
gdino_params = sum(p.numel() for p in gdino_model.parameters())
print(f"   ✅ Done! ({gdino_params/1e6:.1f}M params)")

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
import hashlib
_detection_cache = {}
_sam_embedding_cache = {}

@torch.no_grad()
def detect_objects(image, text_labels, box_threshold=0.3, text_threshold=0.25):
    """
    Use Grounding DINO to detect objects matching the text in an image.

    Args:
        image: PIL Image
        text_labels: list of str (e.g., ["dog", "cat"])
    Returns:
        boxes (N,4), scores (N,), labels list[str]
    """
    key = (hashlib.sha256(image.tobytes()).hexdigest(), image.size, tuple(text_labels))
    if key not in _detection_cache:
        inputs = gdino_processor(images=image, text=[text_labels], return_tensors="pt").to(device)
        _detection_cache[key] = gdino_model(**inputs)
    outputs = _detection_cache[key]
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, threshold=box_threshold, text_threshold=text_threshold,
        target_sizes=[(image.height, image.width)]
    )
    r = results[0]
    return r['boxes'].cpu().numpy(), r['scores'].cpu().numpy(), r['text_labels']

print("✅ detect_objects defined")

In [ ]:
def draw_detections(image, boxes, scores, labels, title='', ax=None):
    if ax is None: fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(image)
    unique = list(set(labels))
    cmap = plt.cm.Set2
    cdict = {l: cmap(i / max(len(unique), 1)) for i, l in enumerate(unique)}
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box
        c = cdict[label]
        rect = mpatches.FancyBboxPatch((x1,y1), x2-x1, y2-y1, linewidth=3,
            edgecolor=c, facecolor='none', boxstyle='round,pad=0')
        ax.add_patch(rect)
        ax.text(x1, y1-8, f'{label}: {score:.2f}', fontsize=11, fontweight='bold',
                color='white', bbox=dict(boxstyle='round,pad=0.3', facecolor=c, edgecolor='none', alpha=0.85))
    ax.set_title(title, fontsize=13, fontweight='bold'); ax.axis('off')
    return ax

print("✅ draw_detections defined")

## Demo code

Change the recorded prompt or threshold. Which objects disappear first?

In [ ]:
image = Image.open(image_paths[0]).convert("RGB")
boxes, scores, labels = detect_objects(image, ["dog", "cat"], box_threshold=0.30)
draw_detections(image, boxes, scores, labels)
plt.show()

In [ ]:
# Test Grounding DINO on Dog/Cat images
sample_indices = [0, 1, NUM_IMAGES_PER_CLASS, NUM_IMAGES_PER_CLASS + 1]
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
fig.suptitle('🎯 Grounding DINO — detecting "dog" and "cat"', fontsize=16, fontweight='bold')

for i, idx in enumerate(sample_indices):
    ax = axes[i // 2, i % 2]
    img = Image.open(image_paths[idx]).convert('RGB')
    boxes, scores, labels = detect_objects(img, ["dog", "cat"], box_threshold=0.25)
    gt = image_labels[idx].upper()
    draw_detections(img, boxes, scores, labels,
                   title=f'GT: {gt} | {len(boxes)} detections', ax=ax)

plt.tight_layout(); plt.show()

In [ ]:
sample_img = Image.open(image_paths[0]).convert('RGB')

prompt_experiments = [
    (["dog"],                   '"dog"'),
    (["animal"],                '"animal"'),
    (["golden retriever"],      '"golden retriever"'),
    (["head", "body", "leg"],   '"head", "body", "leg"'),
]

fig, axes = plt.subplots(1, len(prompt_experiments), figsize=(6*len(prompt_experiments), 6))
fig.suptitle('🔍 Same image, different prompts → different detection results', fontsize=16, fontweight='bold')

for i, (text_labels, name) in enumerate(prompt_experiments):
    boxes, scores, labels = detect_objects(sample_img, text_labels, box_threshold=0.2)
    draw_detections(sample_img, boxes, scores, labels,
                   title=f'Prompt: {name}\n({len(boxes)} detections)', ax=axes[i])

plt.tight_layout(); plt.show()

## Interpretation

Model scores are not calibrated probabilities.

Compare the output with the evidence in the input. A generated answer can be wrong.